# 🌿 Darukaa Reference Benchmarking Pipeline

Profile-first, non-compensatory, evidence-graded biodiversity benchmarking.
This notebook runs the pipeline and displays its outputs; it does **not** recompute a
composite (the pipeline does that internally, correctly). See `METHODOLOGY_MASTER.md`
and `ASSUMPTIONS_AND_LIMITATIONS.md` before quoting any result.

## 1. Setup — clone repo & install

In [8]:
# Clone (or update) the repo and install it.
import os, glob
REPO = 'reference-benchmarking'
if not os.path.exists(REPO):
    !git clone https://github.com/G-auravSingh/reference-benchmarking.git

candidates = sorted(glob.glob(f'{REPO}/darukaa_reference_v*'))
assert candidates, f'No darukaa_reference_v* folder found under {REPO}/ — check the clone succeeded.'
PKG_DIR = candidates[-1]  # highest version string, sorted lexicographically
print('Using package directory:', PKG_DIR)
%cd $PKG_DIR
!pip install -q -r requirements.txt
!pip install -q -e .


Cloning into 'reference-benchmarking'...
remote: Enumerating objects: 1695, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 1695 (delta 88), reused 124 (delta 46), pack-reused 1498 (from 1)
Receiving objects: 100% (1695/1695), 14.22 MiB | 17.21 MiB/s, done.
Resolving deltas: 100% (626/626), done.
Using package directory: reference-benchmarking/darukaa_reference_v0.2.7
/content/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7
  Preparing metadata (setup.py) ... done


## 2. Authenticate Google Earth Engine
Uses your GEE project (e.g. `your-gee-project-id`).

In [9]:
import ee
ee.Authenticate()
#ee.Initialize(project='darukaa-earth-product')  # <- your GEE project id
ee.Initialize(project='gaurav-singh-007')
print('Earth Engine ready')

Earth Engine ready


## 3. Upload your site KML/KMZ
One or more polygons (a single site, or DBSCAN assessment-cluster tiles from the
site-selection pipeline for large agroforestry AOIs).

In [ ]:
from google.colab import files
up = files.upload()
SITE_PATH = list(up.keys())[0]
print('Using:', SITE_PATH)

## 4. Configure the run
Everything is driven by one `Config`. The important v0.2.0 knobs are shown; all have
safe defaults. `reference_stratification` defaults to `ecoregion_landcover` (the
SEED-faithful path) — **verify the PNV crosswalk before trusting a live run** (see
README §6 and `ASSUMPTIONS_AND_LIMITATIONS.md`).

In [ ]:
from darukaa_reference.config import Config

config = Config(
    gee_project='your-gee-project-id',
    output_dir='outputs',
    output_format='both',              # json + csv (+ html always)
    # --- project context (shared with site-selection) ---
    realm='terrestrial', archetype='conservation', assessment_mode='baseline',
    # --- reference (SEED) ---
    hmi_hard_ceiling=0.05,             # SEED maximum
    use_variance_stability_floor=True, # OD-3: suppress score on a noisy reference
    reference_stratification='ecoregion_landcover',  # default, SEED-faithful; see README §6
    # --- optional SEED kernel view (OD-4) ---
    use_seed_kernel=False, seed_kernel_delta=0.5,
)
from darukaa_reference.indicators import create_default_registry
registry = create_default_registry()  # <- builds the 44-indicator contract; needed by Pipeline()

print('archetype=%s mode=%s stratification=%s' % (config.archetype, config.assessment_mode, config.reference_stratification))
print('%d indicators registered, %d scored by default' % (len(registry), len(registry.scored())))

## 4b. Choosing what gets scored (every registered indicator, by pillar)

This is the full picker — everything registered (real count shown live below, not hardcoded here — that number has drifted stale in this exact spot before), grouped by pillar (C1 landscape,
C2 vegetation, C3 fauna, C4 pressure), with the Darukaa-recommended default already
marked. Read this table here — you don't need to open any other document to decide
what to score for this project. Edit the two lists in the next cell to change
anything, in **either direction**: activate something beyond the default, or turn
off one of the defaults you don't want for this project.


In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 90)

avail = pd.DataFrame(registry.availability_table())
avail['recommended_default'] = avail['currently_scored'] & ~avail['client_override']
cols = ['name','construct','display_name','disposition','recommended_default',
        'currently_scored','evidence_tier','why']
avail = avail[cols].sort_values(['construct','name'], na_position='last')
avail.style.apply(lambda r: ['background-color:#e6f4ec' if r['recommended_default']
                             else '' for _ in r], axis=1)

**Disposition meanings** (see `contracts.py` for the full reasoning per indicator):
- `retain`/`redefine` — scored by default (the Darukaa-recommended set).
- `context` — not scored by default, usually redundancy/parsimony, not a construct
  flaw. **Freely activatable.**
- `screening` — has a real, documented scoring limitation. **Activatable only with
  an explicit force**, and the caveat travels with it permanently in the report.
- `remove` — a documented, unrepairable flaw (e.g. CERI). **Never activatable.**

In [ ]:
# Edit these lists, then run this cell. Leave all empty to keep the recommended default.
ACTIVATE_ADDITIONAL_INDICATORS = []   # e.g. ['natural_landcover', 'ndvi']
FORCE_ACTIVATE_SCREENING = []         # subset of the above needing an explicit force
DEACTIVATE_DEFAULT_INDICATORS = []    # e.g. ['ghm'] — turn OFF a recommended default

from darukaa_reference import contracts

if DEACTIVATE_DEFAULT_INDICATORS:
    for name, r in contracts.request_deactivation(registry, DEACTIVATE_DEFAULT_INDICATORS).items():
        print(f"{name}: {r['status']}\n  {r['reason']}\n")

if ACTIVATE_ADDITIONAL_INDICATORS:
    for name, r in contracts.request_activation(
            registry, ACTIVATE_ADDITIONAL_INDICATORS, force=FORCE_ACTIVATE_SCREENING).items():
        print(f"{name}: {r['status']}\n  {r['reason']}\n")

print('Final scored set (%d), by pillar:' % len(registry.scored()))
from collections import defaultdict
by_pillar = defaultdict(list)
for s in registry.scored():
    by_pillar[s.construct].append(s.name)
for pillar in ['C1_landscape','C2_vegetation','C3_fauna','C4_pressure']:
    print(f'  {pillar:16s}: {by_pillar.get(pillar, [])}')

## 5. Run the pipeline
Loads sites → resolves ecoregions → builds SEED references → benchmarks each indicator
→ profile-first non-compensatory scoring → writes `outputs/benchmark_scorecard.json/.csv/.html`.
Removed/inactive indicators (e.g. CERI) are skipped automatically.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
from darukaa_reference.pipeline import Pipeline

report = Pipeline(config, registry).run(site_path=SITE_PATH)  # registry from Section 4
print('\nStatus:', {k: len(v) for k, v in report['indicator_status'].items()})

## 6. What is scored, and what is not
Every indicator's honest disposition — nothing hidden.

In [ ]:
import pandas as pd
st = report['indicator_status']
for k in ['scored','contextual','screening_only','pending_inputs','removed']:
    print(f"{k:16s} ({len(st.get(k,[]))}): {', '.join(st.get(k,[])) or '—'}")

## 7. Site profiles (profile-first)
The primary output: per-component limiting factor, the condition × pressure decision,
and the secondary roll-up with its stability flag. Read the minimum component first.

In [ ]:
# REAL FIX: this cell used to print raw, unbounded scores (e.g. 'C1_landscape
# 0.297') straight from the profile dict for each site -- the same class of
# confusing output the multi-tile project cell had (now fixed, see Section 11).
# Uses the same real son_score functions the HTML report itself uses, so this
# quick preview is consistent with the full report below, not a stale, separate
# view of the same data.
from darukaa_reference import son_score

for site_id, prof in report.get('site_profiles', {}).items():
    site_rows = [r for r in report['scorecard'] if r.get('site_id') == site_id]
    summary = son_score.son_summary(prof, site_rows, son_score.PILLAR_NAMES)
    oc, op = summary['overall_condition'], summary['overall_pressure']
    chain = summary['limiting_chain']
    chain_str = chain['display'] if chain['available'] else 'no pillar had scored data this run'
    chain_display = chain_str[0].upper() + chain_str[1:]  # NOT .capitalize() -- that
    # lowercases 'C1' etc; see html_report.py's own real bug fix for why.
    print(f"\n=== {site_id} — decision: {summary['matrix_cell']} ===")
    print(f"  Overall SoN: {oc['score_pct']} {oc['concern_class']}  ({chain_display})")
    for p in summary['pillars']:
        limiting = ' & '.join(p['limiting_indicators']) if p['limiting_indicators'] else p['limiting_subdimension']
        print(f"  {p['pillar_label']:32s} {p['score_pct']:>5s}  {p['concern_class']:10s} (limited by: {limiting})")
    print(f"  Pressure: {op['score_pct']} {op['concern_class']}  (kept structurally separate)")


## 8. Indicator scorecard

In [ ]:
df = pd.DataFrame(report['scorecard'])
cols = [c for c in ['site_id','indicator','construct','evidence_tier',
        'site_value','tier2_benchmark','tier2_benchmark_estimator',
        'tier2_display_pct_of_reference','reference_type'] if c in df.columns]
df[cols]

## 9. Evidence-graded HTML report
The client deliverable — a deterministic projection of the registry, generated every run.

In [ ]:
from IPython.display import HTML, FileLink
html_path = 'outputs/benchmark_scorecard.html'
display(FileLink(html_path))
HTML(open(html_path).read())

## 9b. Maps (optional)

For any scored indicator whose underlying data is a geospatial raster in GEE
(most of them), this renders it as an interactive map, clipped to the site with a
buffer for context. Uses `geemap` (pre-installed in Colab; installs automatically
below if it's ever missing). Indicators without a mapped GEE image (e.g. some
aquatic/derived metrics) are skipped and listed, not silently omitted.

In [ ]:
try:
    import geemap
except ImportError:
    !pip install -q geemap
    import geemap

import ee

def show_indicator_maps(registry, config, site_geometry, indicator_names=None, buffer_km=10):
    """Render one geemap.Map per indicator with a live GEE image, clipped to the
    site + a buffer for landscape context. site_geometry: an ee.Geometry."""
    names = indicator_names or [s.name for s in registry.scored()]
    region = site_geometry.buffer(buffer_km * 1000)
    skipped = []
    for name in names:
        spec = registry.get(name) if name in registry else None
        img_fn = spec.metadata.get('gee_image_fn') if spec else None
        if img_fn is None:
            skipped.append(name); continue
        try:
            img = img_fn(config)
            if img is None:
                skipped.append(name); continue
            m = geemap.Map()
            m.centerObject(region, zoom=12)
            m.addLayer(img.clip(region), {}, spec.display_name)
            m.addLayer(site_geometry, {'color': 'red'}, 'site boundary', opacity=0.6)
            print(f"--- {spec.display_name} ({name}) ---")
            display(m)
        except Exception as e:
            print(f"  {name}: map unavailable ({e})")
            skipped.append(name)
    if skipped:
        print('No mapped GEE layer for:', skipped)

# Uses the site geometry loaded in Section 3. Adjust indicator_names to a subset
# if you only want specific maps, e.g. indicator_names=['natural_habitat','bii'].
import geopandas as gpd
_sites_gdf = gpd.read_file(SITE_PATH)
_site_geom = ee.Geometry(_sites_gdf.geometry.unary_union.__geo_interface__)
show_indicator_maps(registry, config, _site_geom)

## 10. Download results

In [ ]:
from google.colab import files
for ext in ['json','csv','html']:
    p = f'outputs/benchmark_scorecard.{ext}'
    if os.path.exists(p): files.download(p)

---
## 11. Multi-tile / multi-zone projects (alternative to steps 3-10 above)

Use this section instead of steps 3-10 when your project has many real zones/EMUs
or scattered parcels (Tata Motors' 9 zones, Soulforest's 7 EMUs, agroforestry
projects) rather than one compact single-boundary site. Each **tile** is dissolved
into one geometry automatically. The project-level result is combined
**non-compensatorily**: the project's signal for every indicator is set by its
**worst tile/zone**, named explicitly — never averaged away by a larger, better one.

**If your project already has a site-selection pipeline run in this same repo**
(Tata Motors, Soulforest, GV, Soova): no upload needed at all — the real tiles are
already on disk from the clone in Step 1. Just set `PROJECT_NAME` below to the real
project folder name and this section finds them automatically.

In [10]:
# REAL CONNECTION: if this project already has a site-selection pipeline run
# pushed into this same repo (Tata Motors, Soulforest, GV, Soova), its real tiles
# are already on disk after the Step 1 clone — no manual upload needed at all.
# Cell 2 already %cd'd into PKG_DIR (darukaa_reference_v0.2.7/), so the real
# repo root (containing BOTH pipelines) is its parent directory.
import sys, os
sys.path.insert(0, os.getcwd())
from run_project_from_manifest import find_manifest_by_project_name, load_manifest, resolve_tile_paths

PROJECT_NAME = 'TataMotors_Pimpri'  # <- CHANGE THIS (e.g. SoulForest_Veltoor, FCF_GV, FCF_Soova)
COMBINE_AQUATIC = True  # <- set False to force a standalone terrestrial-only run
                        # even when a real aquatic companion manifest exists
REPO_ROOT = os.path.dirname(os.getcwd())

try:
    manifest_path = find_manifest_by_project_name(REPO_ROOT, PROJECT_NAME)
    manifest = load_manifest(manifest_path)
    TILE_PATHS = resolve_tile_paths(manifest, manifest_path)
    TILE_LABELS = list(manifest['tile_labels'])
    is_already_aquatic = 'aquatic' in PROJECT_NAME.lower()
    TILE_REALMS = ['aquatic' if is_already_aquatic else 'terrestrial'] * len(TILE_PATHS)
    print(f'Found real manifest at: {manifest_path}')
    print(f'{len(TILE_PATHS)} real tiles for {PROJECT_NAME}')

    # REAL COMBINING ("if any project involves both
    # aquatic + terrestrial the report can't be a separate one") — auto-finds
    # a real '<PROJECT_NAME>_Aquatic' companion manifest and merges it in, each
    # tile keeping its own correct realm. Only attempted from the terrestrial
    # side (running an already-aquatic project alone stays standalone).
    if COMBINE_AQUATIC and not is_already_aquatic:
        aquatic_name = f'{PROJECT_NAME}_Aquatic'
        try:
            aq_manifest_path = find_manifest_by_project_name(REPO_ROOT, aquatic_name)
            aq_manifest = load_manifest(aq_manifest_path)
            aq_tile_paths = resolve_tile_paths(aq_manifest, aq_manifest_path)
            print(f'Found real aquatic companion "{aquatic_name}" ({len(aq_tile_paths)} tile(s)) -- combining.')
            TILE_PATHS += aq_tile_paths
            TILE_LABELS += aq_manifest['tile_labels']
            TILE_REALMS += ['aquatic'] * len(aq_tile_paths)
        except FileNotFoundError:
            pass  # no real aquatic companion for this project -- stays terrestrial-only

    print(f'\nFinal tile list ({len(TILE_PATHS)} real tile(s)):')
    for p, l, r in zip(TILE_PATHS, TILE_LABELS, TILE_REALMS):
        print(f'  [{r}] {l}  <-  {p}')
except FileNotFoundError:
    # No site-selection pipeline run for this project in this repo — fall back
    # to manual upload (e.g. a project with no site-selection pipeline at all,
    # like Corbett's raw KML sites).
    print(f'No site-selection handoff found for "{PROJECT_NAME}" in this repo — '
          f'falling back to manual upload.')
    from google.colab import files
    up = files.upload()
    TILE_PATHS = list(up.keys())
    TILE_LABELS = [p.rsplit('.', 1)[0] for p in TILE_PATHS]
    TILE_REALMS = ['terrestrial'] * len(TILE_PATHS)  # adjust manually if this is actually an aquatic site


Found real manifest at: /content/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/Darukaa_SiteSelection_pipeline_vAug2026/Darukaa_SiteSelection_pipeline/Darukaa_SiteSelection/projects/TataMotors_Pimpri/outputs/07_reference_handoff/tile_manifest.json
9 real tiles for TataMotors_Pimpri
Found real aquatic companion "TataMotors_Pimpri_Aquatic" (6 tile(s)) -- combining.

Final tile list (15 real tile(s)):
  [terrestrial] EMU_Deccan_forest  <-  /content/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/Darukaa_SiteSelection_pipeline_vAug2026/Darukaa_SiteSelection_pipeline/Darukaa_SiteSelection/projects/TataMotors_Pimpri/outputs/07_reference_handoff/tiles/TataMotors_Pimpri_EMU_Deccan_forest.geojson
  [terrestrial] EMU_Grass_land  <-  /content/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmark

In [11]:
# Same Config as the single-site flow — just set archetype='agroforestry'.
# If you already ran Section 4 above for a single-site run, you can reuse
# that `config` object instead of rebuilding it here.
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry

config = Config(
    gee_project='gaurav-singh-007',
    output_dir='outputs',
    archetype='industrial',  # 'industrial' for Tata Motors, 'conservation' for
    # Soulforest, 'agroforestry' for GV/Soova — match the real site-selection
    # project's own config.yaml archetype, not guessed
    realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True,
    reference_stratification='ecoregion_landcover',
)
registry = create_default_registry()

*Want to score something beyond the default set for this project? Same mechanism as Section 4b above — run it here against this `registry` before Section 11's pipeline-run cell:*

In [12]:
# REAL, COMPLETE indicator picker -- every one of the 45 registered
# indicators, grouped by real pillar (C1 landscape / C2 vegetation /
# C3 fauna / C4 pressure), with its current status shown BEFORE you
# choose anything, so you never need to leave this notebook or check
# an external doc. 'scored' = drives the condition/pressure score today;
# everything else is real, computed context -- pick any of those into
# ACTIVATE_ADDITIONAL_INDICATORS below to make them scored too, in
# whichever pillar they already belong to (you don't choose the pillar --
# each indicator's pillar is fixed by what it actually measures; you're
# choosing WHICH indicators join the score, not reassigning them).
PILLAR_NAMES = {'C1_landscape': 'C1 -- Landscape extent', 'C2_vegetation': 'C2 -- Vegetation condition',
                'C3_fauna': 'C3 -- Faunal condition', 'C4_pressure': 'C4 -- Pressures & human interface'}
STATUS_LABEL = {'baseline': 'SCORED', 'monitoring': 'SCORED', 'contextual': 'context (not scored)',
                'screening': 'screening-only (never scored -- real, known limitation)',
                'pending': 'pending (a dependency is unmet)', 'removed': 'removed (de-scoped)'}
for construct in ['C1_landscape', 'C2_vegetation', 'C3_fauna', 'C4_pressure']:
    print(f"\n=== {PILLAR_NAMES[construct]} ===")
    members = [s for s in registry.all() if s.construct == construct]
    for s in sorted(members, key=lambda s: (not s.scoring_eligible, s.name)):
        realms = ', '.join(s.applicable_realms)
        print(f"  {s.name:28} [{STATUS_LABEL.get(s.evidence_tier, s.evidence_tier):45}] "
              f"realms=({realms})")
        if s.evidence_tier == 'screening':
            print(f"    -- will never be scored even if activated below: {s.citation[:100] if s.citation else ''}")
print(f"\n{len(registry.scored())} of {len(registry.all())} indicators are scored by default: "
      f"{[s.name for s in registry.scored()]}")

# Same bidirectional mechanism as Section 4b (single-site). Edit and run if needed --
# use the real 'name' shown above (e.g. 'natural_landcover'), not display_name, in
# these three lists. A name's real pillar (shown above) is where it lands once scored --
# you cannot move an indicator to a different pillar, only choose to score it or not.
ACTIVATE_ADDITIONAL_INDICATORS = []
FORCE_ACTIVATE_SCREENING = []  # real screening-tier indicators genuinely cannot be
                               # trusted as a score (see the reason printed above each
                               # one) -- only list a name here if you've read that reason
                               # and want it scored anyway, fully informed
DEACTIVATE_DEFAULT_INDICATORS = []

from darukaa_reference import contracts
if DEACTIVATE_DEFAULT_INDICATORS:
    for name, r in contracts.request_deactivation(registry, DEACTIVATE_DEFAULT_INDICATORS).items():
        print(f"{name}: {r['status']}\n  {r['reason']}\n")
if ACTIVATE_ADDITIONAL_INDICATORS:
    for name, r in contracts.request_activation(
            registry, ACTIVATE_ADDITIONAL_INDICATORS, force=FORCE_ACTIVATE_SCREENING).items():
        print(f"{name}: {r['status']}\n  {r['reason']}\n")
print('\nFinal scored set (%d):' % len(registry.scored()), [s.name for s in registry.scored()])



=== C1 -- Landscape extent ===
  cpland                       [SCORED                                       ] realms=(terrestrial, mixed)
  flii                         [SCORED                                       ] realms=(terrestrial, mixed)
  forest_loss_rate             [SCORED                                       ] realms=(terrestrial, mixed)
  jrc_water_persistence        [SCORED                                       ] realms=(terrestrial, aquatic, mixed)
  natural_habitat              [SCORED                                       ] realms=(terrestrial, mixed)
  kba_overlap                  [screening-only (never scored -- real, known limitation)] realms=(terrestrial, aquatic, mixed)
    -- will never be scored even if activated below: IUCN (2022). KBA Standards. DOI:10.2305/IUCN.CH.2022.24.en
  natural_landcover            [context (not scored)                         ] realms=(terrestrial, mixed)
  rci                          [context (not scored)                         ] 

In [13]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
from darukaa_reference.project_aggregation import run_multi_tile_project

# PROJECT_NAME, TILE_PATHS, TILE_LABELS, TILE_REALMS already set above (cell 28).
project = run_multi_tile_project(
    config, registry,
    tile_paths=TILE_PATHS, tile_labels=TILE_LABELS,
    project_name=PROJECT_NAME, tile_realms=TILE_REALMS,
)
print('\nStatus:', {k: len(v) for k, v in project['indicator_status'].items()})
print('Tiles: %d succeeded, %d failed. Total area: %.1f ha' % (
    project['meta']['n_tiles_succeeded'], project['meta']['n_tiles_failed'],
    project['meta']['total_area_ha']))
if project['meta']['failed_tiles']:
    print('FAILED TILES:', project['meta']['failed_tiles'])



Status: {'scored': 12, 'contextual': 32, 'screening_only': 6, 'pending_inputs': 1, 'removed': 1}
Tiles: 15 succeeded, 0 failed. Total area: 82.3 ha


### Why the project profile looks like this — per-indicator worst tile

In [14]:
# REAL FIX: this cell used to show a bare table of raw, unbounded worst-tile
# z-score-like values with no intactness %, no concern level, and no limiting
# chain -- exactly the confusing presentation the full report below has since
# been rebuilt to fix. Updated to show the SAME real, bounded %/concern-level
# numbers the full HTML report uses (scoring.normalize + son_score.classify),
# so this quick preview is consistent with it, not a stale, separate view.
import pandas as pd
from darukaa_reference import scoring, son_score

def _as_pct(score):
    return 'N/A' if score is None else f'{round(score * 100)}%'

rows = []
for name, s in project['multi_tile_summary']['per_indicator'].items():
    if s.get('status') == 'ok':
        bounded = scoring.normalize(s['worst_tile_benchmark'], s.get('tier2_benchmark_estimator', 'robust_z'))
        rows.append({
            'indicator': name, 'worst_tile': s['worst_tile'],
            'intactness_pct': _as_pct(bounded),
            'concern': son_score.classify(bounded, son_score.DEFAULT_BANDS),
            'tiles_with_data': f"{s['n_tiles_with_data']}/{s['n_tiles_total']}"})
print('Quick preview — the full report below (cell 38) has the complete picture: '
      'per-zone raw values, pillar cards, and the full limiting chain.')
pd.DataFrame(rows)


Quick preview — the full report below (cell 38) has the complete picture: per-zone raw values, pillar cards, and the full limiting chain.


,indicator,worst_tile,intactness_pct,concern,tiles_with_data
0,natural_habitat,EMU_Trail_plots,43%,Moderate,9/15
1,flii,EMU_Trail_plots,0%,Very High,9/15
2,eii,EMU_Wildlife,6%,Very High,9/15
3,bii,EMU_Wetland_forest,0%,Very High,9/15


### Project-level profile (same profile-first output as a single site)

In [15]:
# REAL FIX: this cell used to print raw, unbounded scores (e.g. 'C1_landscape
# 0.000') straight from the profile dict -- exactly the confusing output that
# prompted the full report rebuild. Updated to use the SAME real son_score
# functions (limiting_chain, pillar_summary) the HTML report itself uses, so
# this quick preview shows the real 1-100% scores, concern levels, and the
# full traceable limiting chain -- consistent with the report, not a stale,
# separate view of the same data.
from darukaa_reference import son_score

prof = project['site_profiles']['PROJECT']
proj_rows = [{'indicator': n, **s} for n, s in project['multi_tile_summary']['per_indicator'].items()]
summary = son_score.son_summary(prof, proj_rows, son_score.PILLAR_NAMES)

oc = summary['overall_condition']
print(f"Overall SoN: {oc['score_pct']} {oc['concern_class']}")
chain_str = summary['limiting_chain']['display'] if summary['limiting_chain']['available'] else ''
# Real bug already caught once in html_report.py: str.capitalize() lowercases
# everything after the first character, mangling 'C1' etc. Only the true
# first character needs capitalising.
chain_display = (chain_str[0].upper() + chain_str[1:]) if chain_str else 'No pillar had scored data this run.'
print(f'  {chain_display}')
print()
for p in summary['pillars']:
    limiting = ' & '.join(p['limiting_indicators']) if p['limiting_indicators'] else p['limiting_subdimension']
    print(f"  {p['pillar_label']:32s} {p['score_pct']:>5s}  {p['concern_class']:10s} (limited by: {limiting})")
print()
op = summary['overall_pressure']
print(f"Pressure axis: {op['score_pct']} {op['concern_class']}  (kept structurally separate — see AGGREGATION_WALKTHROUGH.md)")
print(f"Matrix cell: {summary['matrix_cell']}")


Overall SoN: 0% Very High
  Limited primarily by C1 — Landscape extent (0%), itself limited by Flii

  C1 — Landscape extent               0%  Very High  (limited by: Flii)
  C2 — Vegetation condition           6%  Very High  (limited by: Eii)
  C3 — Faunal condition               0%  Very High  (limited by: Bii)

Pressure axis: N/A None  (kept structurally separate — see AGGREGATION_WALKTHROUGH.md)
Matrix cell: insufficient_data


### Project-level evidence-graded HTML report

In [16]:
from IPython.display import HTML, FileLink
html_path = f'outputs/{PROJECT_NAME}_project.html'
display(FileLink(html_path))
HTML(open(html_path).read())

/content/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7/reference-benchmarking/darukaa_reference_v0.2.7/outputs/TataMotors_Pimpri_project.html

### Download everything (project-level + every individual tile's own report)

In [17]:
from google.colab import files
import shutil, os

# Zip the whole outputs folder so project-level AND per-tile files come as one download.
shutil.make_archive(f'{PROJECT_NAME}_results', 'zip', 'outputs')
files.download(f'{PROJECT_NAME}_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Monitoring mode (cycle 2+)
Set `assessment_mode='monitoring'`, run this cycle, then diff against the stored Year-0
report with `change.py`:


In [ ]:
# from darukaa_reference import change as CH
# import json
# base = CH.from_report(json.load(open('outputs/benchmark_scorecard_YEAR0.json')))
# cur  = CH.from_report(report)
# res  = CH.score_cycle(base, cur)
# print(res['summary'])  # per-indicator change + BACI where controls exist

### Adding a custom indicator
Register with the full contract so eligibility is computed correctly (see `INDICATOR_REGISTER.md`).

In [ ]:
# from darukaa_reference.registry import IndicatorRegistry
# r = create_default_registry()
# r.register(name='my_metric', display_name='My Metric', source_type='gee',
#            extract_fn=my_fn, construct='C2_vegetation', subdimension='structure',
#            measurement_scale='ratio', evidence_tier='baseline',
#            reference_type='contemporary_best_on_offer', uncertainty_method='bootstrap_ci',
#            input_layers=['my_layer'])